# Data preprocessing for field validation

In [1]:
import sys
import pandas as pd  
import matplotlib.pyplot as plt
sys.path.append('../../../')   # Add parent directory to Python path
from utils.visualization import *

## Write Video data labels for ground truth in video data

In [ ]:
# read the label CSV file
df_label = pd.read_csv('../../../data/field_validation/grund_truth_labels.csv')
df_video = pd.read_csv('../../../data/field_validation/video.csv')

,Label,start_timestamp,end_timestamp
0,1,"00:04:26,241","00:04:27,508"
1,5,"00:04:31,509","00:04:32,508"
2,1,"00:04:39,509","00:04:40,262"
3,5,"00:04:41,260","00:04:42,760"
4,4,"00:04:44,511","00:04:48,761"
5,1,"00:04:51,765","00:04:52,766"
6,5,"00:04:53,513","00:04:54,759"
7,1,"00:05:03,515","00:05:05,013"
8,7,"00:10:15,794","00:10:17,794"
9,1,"00:12:29,294","00:12:31,291"


In [41]:
# df_label, for each page, from start_timestamp	to end_timestamp, if column Video in df_video, in the range from  start_timestamp	to end_timestamp, then give the extra label from df_label
def assign_labels_to_video(df_video, df_label):
    """
    Assign labels from df_label to df_video based on timestamp ranges.
    
    Parameters:
    -----------
    df_video : pd.DataFrame
        DataFrame with video data containing a 'Video' timestamp column
    df_label : pd.DataFrame
        DataFrame with labels containing 'start_timestamp', 'end_timestamp', 
        and label columns
    
    Returns:
    --------
    pd.DataFrame
        df_video with additional label columns from df_label
    """
    # Create a copy to avoid modifying original
    df_result = df_video.copy()
    count = 0
    countmask = 0
    
    # Ensure timestamps are datetime objects
    df_result['Video'] = pd.to_datetime(df_result['Video'])
    df_label['start_timestamp'] = pd.to_datetime(df_label['start_timestamp'])
    df_label['end_timestamp'] = pd.to_datetime(df_label['end_timestamp'])
    
    # Get label columns (exclude timestamp columns)
    label_columns = [col for col in df_label.columns 
                     if col not in ['start_timestamp', 'end_timestamp']]
    print(label_columns)
    
    # Initialize label columns in result DataFrame
    for col in label_columns:
        df_result[col] = 0
    
    # Iterate through each row in df_label
    for idx, label_row in df_label.iterrows():
        start = label_row['start_timestamp']
        end = label_row['end_timestamp']
        
        # Find matching rows in df_video where video timestamp is in range
        mask = (df_result['Video'] >= start) & (df_result['Video'] <= end)
        
        # Assign labels to matching rows
        for col in label_columns:
            df_result.loc[mask, col] = label_row[col]
            count += 1
            countmask += mask.sum()

    print(f"Total labels assigned: {count}")
    print(f"Total rows updated: {countmask}")
    return df_result



In [ ]:
# Apply the method
df_video_labeled = assign_labels_to_video(df_video, df_label)


C:\Users\liuzi\AppData\Local\Temp\ipykernel_14600\1615381946.py:25: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_result['Video'] = pd.to_datetime(df_result['Video'])


['Label']
Total labels assigned: 47
Total rows updated: 25837


In [46]:
print_sampling_frequency(df_video_labeled)
# Convert timestamps to datetime if not already
df_label['start_timestamp'] = pd.to_datetime(df_label['start_timestamp'])
df_label['end_timestamp'] = pd.to_datetime(df_label['end_timestamp'])

# Calculate duration in seconds
df_label['duration_seconds'] = (df_label['end_timestamp'] - df_label['start_timestamp']).dt.total_seconds()

# Display the results
print(f"Total duration: {df_label['duration_seconds'].sum()} seconds")

Sampling frequency: 99.35 Hz
Total duration: 260.0 seconds


In [50]:
df_video_labeled.head(10)

,NTP,Timestamp,Video,X,Y,Z,Label
0,2025-10-21 15:15:42.041,"2025-10-21, 15:15:42.0390",2025-11-04 00:00:00.104,-0.161850,-0.621307,-0.760971,0
1,2025-10-21 15:15:42.042,"2025-10-21, 15:15:42.0410",2025-11-04 00:00:00.105,-0.166687,-0.623611,-0.760941,0
2,2025-10-21 15:15:42.049,"2025-10-21, 15:15:42.0480",2025-11-04 00:00:00.112,-0.161774,-0.620071,-0.765106,0
3,2025-10-21 15:15:42.052,"2025-10-21, 15:15:42.0500",2025-11-04 00:00:00.114,-0.156738,-0.621979,-0.765808,0
4,2025-10-21 15:15:42.053,"2025-10-21, 15:15:42.0520",2025-11-04 00:00:00.116,-0.152634,-0.622070,-0.766769,0
5,2025-10-21 15:15:42.056,"2025-10-21, 15:15:42.0550",2025-11-04 00:00:00.119,-0.145859,-0.618652,-0.768005,0
6,2025-10-21 15:15:42.060,"2025-10-21, 15:15:42.0580",2025-11-04 00:00:00.122,-0.150040,-0.623901,-0.762604,0
7,2025-10-21 15:15:42.061,"2025-10-21, 15:15:42.0600",2025-11-04 00:00:00.124,-0.149002,-0.620865,-0.763474,0
8,2025-10-21 15:15:42.067,"2025-10-21, 15:15:42.0650",2025-11-04 00:00:00.130,-0.152893,-0.623749,-0.761902,0
9,2025-10-21 15:15:42.074,"2025-10-21, 15:15:42.0720",2025-11-04 00:00:00.136,-0.153412,-0.622101,-0.762039,0


In [ ]:
df_accelerometer = pd.read_csv('../../../data/field_validation/accelerometer.csv')
df_accelerometer.head()

,Date,NTP,GNSS-Time,Acc-X,Acc-Y,Acc-Z
0,2025-10-21T15:15:40.982,2025-10-21 15:15:40.355,-1,-1.877060,6.095642,5.679047
1,2025-10-21T15:15:40.986,2025-10-21 15:15:40.360,-1,0.110138,6.258438,6.655884
2,2025-10-21T15:15:40.991,2025-10-21 15:15:40.365,-1,2.231400,6.584061,7.422028
3,2025-10-21T15:15:41,2025-10-21 15:15:40.374,-1,3.045425,6.833054,7.599197
4,2025-10-21T15:15:41.002,2025-10-21 15:15:40.375,-1,2.686295,6.823471,7.129929


In [ ]:
# if NTP in df_video_labeled and Label != 0, 
# then get the corresponding biggest and smallest NTP in df_accelerometer, 
# and fill the rows between these two timestamps with the label value from df_video_labeled 


# Relabel for better implementation

In [51]:
# read the label CSV file
df_label = pd.read_csv('../../../data/field_validation/grund_truth_labels.csv')
print(df_label.head())
df_video = pd.read_csv('../../../data/field_validation/video.csv')
print(df_video.head())

   Label start_timestamp end_timestamp
0      1    00:04:26,241  00:04:27,508
1      5    00:04:31,509  00:04:32,508
2      1    00:04:39,509  00:04:40,262
3      5   00:04:41,260   00:04:42,760
4      4    00:04:44,511  00:04:48,761
                         NTP                  Timestamp         Video  \
0  2025-10-21, 15:15:42.0410  2025-10-21, 15:15:42.0390  00:00:00.104   
1  2025-10-21, 15:15:42.0420  2025-10-21, 15:15:42.0410  00:00:00.105   
2  2025-10-21, 15:15:42.0490  2025-10-21, 15:15:42.0480  00:00:00.112   
3  2025-10-21, 15:15:42.0520  2025-10-21, 15:15:42.0500  00:00:00.114   
4  2025-10-21, 15:15:42.0530  2025-10-21, 15:15:42.0520  00:00:00.116   

          X         Y         Z  
0 -0.161850 -0.621307 -0.760971  
1 -0.166687 -0.623611 -0.760941  
2 -0.161774 -0.620071 -0.765106  
3 -0.156738 -0.621979 -0.765808  
4 -0.152634 -0.622070 -0.766769  


In [ ]:
# for each row in df_label, 
# 1. for each start_timestamp get corresponding NTP in df_video, and an extra colum with name Label_new from label colum
# 2. for each end_timestamp get corresponding NTP in df_video, and an extra colum with name Label_new= 0

In [54]:
def create_label_transitions(df_video, df_label):
    """
    Create label transition points by mapping start/end timestamps from df_label 
    to corresponding NTP values in df_video.
    
    For each row in df_label:
    - At start_timestamp: create entry with NTP and Label_new = label value
    - At end_timestamp: create entry with NTP and Label_new = 0
    
    Parameters:
    -----------
    df_video : pd.DataFrame
        DataFrame with video data containing 'Video' and 'NTP' columns
    df_label : pd.DataFrame
        DataFrame with labels containing 'start_timestamp', 'end_timestamp', 
        and 'Label' columns
    
    Returns:
    --------
    pd.DataFrame
        DataFrame with NTP timestamps and corresponding Label_new values
    """
    # Ensure timestamps are datetime objects
    df_video['Video'] = pd.to_datetime(df_video['Video'])
    df_label['start_timestamp'] = pd.to_datetime(df_label['start_timestamp'])
    df_label['end_timestamp'] = pd.to_datetime(df_label['end_timestamp'])
    
    # List to store transition points
    transitions = []
    start_count = 0
    end_count = 0
    
    # Iterate through each row in df_label
    for idx, label_row in df_label.iterrows():
        start_time = label_row['start_timestamp']
        end_time = label_row['end_timestamp']
        label_value = label_row['Label']
        
        # Find closest NTP in df_video for start_timestamp
        start_mask = df_video['Video'] >= start_time
        if start_mask.any():
            start_ntp = df_video.loc[start_mask, 'NTP'].iloc[0]
            transitions.append({
                'NTP': start_ntp,
                'Label_new': label_value
            })
            start_count += 1
        
        # Find closest NTP in df_video for end_timestamp
        end_mask = df_video['Video'] >= end_time
        if end_mask.any():
            end_ntp = df_video.loc[end_mask, 'NTP'].iloc[0]
            transitions.append({
                'NTP': end_ntp,
                'Label_new': 0
            })
            end_count += 1
    
    # Create DataFrame from transitions
    df_transitions = pd.DataFrame(transitions)
    
    # Sort by NTP timestamp
    df_transitions = df_transitions.sort_values('NTP').reset_index(drop=True)
    
    return df_transitions

In [ ]:
# Apply the method
df_label_transitions = create_label_transitions(df_video, df_label)


,NTP,Label_new
0,"2025-10-21, 15:20:07.9440",1
1,"2025-10-21, 15:20:08.9410",0
2,"2025-10-21, 15:20:12.9370",5
3,"2025-10-21, 15:20:13.9440",0
4,"2025-10-21, 15:20:20.9400",1
5,"2025-10-21, 15:20:21.9470",0
6,"2025-10-21, 15:20:22.9430",5
7,"2025-10-21, 15:20:23.9400",0
8,"2025-10-21, 15:20:25.9430",4
9,"2025-10-21, 15:20:29.9390",0


In [56]:
df_label_transitions.to_csv('../../../data/field_validation/label_for_accelerometer.csv', index=False)